In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Transformer encoder block bằng NumPy

Luồng: positional encoding → self-attention + residual + norm → FFN + residual + norm.

In [ ]:
def softmax(x):
    e=np.exp(x-x.max(-1,keepdims=True)); return e/e.sum(-1,keepdims=True)
def layer_norm(x,eps=1e-5): return (x-x.mean(-1,keepdims=True))/np.sqrt(x.var(-1,keepdims=True)+eps)
def positional_encoding(length,d_model):
    pos=np.arange(length)[:,None]; idx=np.arange(0,d_model,2); rates=np.exp(-math.log(10000)*idx/d_model)
    pe=np.zeros((length,d_model)); pe[:,0::2]=np.sin(pos*rates); pe[:,1::2]=np.cos(pos*rates); return pe
def encoder_block(x,wq,wk,wv,wo,w1,w2):
    q=x@wq; k=x@wk; v=x@wv; a=softmax(q@k.T/math.sqrt(q.shape[-1]))@v
    h=layer_norm(x+a@wo); ff=np.maximum(0,h@w1)@w2; return layer_norm(h+ff)
rng=np.random.default_rng(42); L,D=5,8; x=rng.normal(size=(L,D))+positional_encoding(L,D)
weights=[rng.normal(0,.2,size=s) for s in [(D,D),(D,D),(D,D),(D,D),(D,16),(16,D)]]
out=encoder_block(x,*weights); assert out.shape==(L,D); assert np.allclose(out.mean(-1),0,atol=1e-6)
print(out.shape,out[0])